# Improving the Employability of IT Students
## Research Results, Data Analysis, and Scientific Visualization

This notebook walks through the full analysis: **ingestion and filtering → statistical aggregation → visualization and export**. All calculations are implemented in `src/` (`process_data.py`, `statistical_analysis.py`, `visualize.py`); the notebook imports them, so the notebook, the scripts and the committed outputs always agree.

**Data limitation.** The dataset holds 17 real, de-identified survey responses (11 applicants). Results are exploratory; with this sample size the hypotheses H1–H3 can be neither confirmed nor rejected.

**Reproducibility.** Run the notebook with *Kernel → Restart & Run All*. Every random step uses seed `42`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

# Project root is the parent of the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import process_data, statistical_analysis, visualize

# Fixed randomness: bootstrap and jitter use this seed everywhere.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("Project root folder:", PROJECT_ROOT.name)
print("Random seed:", RANDOM_SEED)

## 1. Data ingestion and filtering

The raw file `data/survey/survey_clean.csv` contains per-student survey-style counts, not timed benchmark runs. There is therefore **no warm-up phase to discard**. The filtering step that plays the same role is the validation in `process_data.clean_data`: it drops records with a missing ID, duplicate IDs, missing or negative counts, and records where interview invitations or offers exceed applications.

In [ ]:
raw = process_data.load_data(process_data.INPUT_FILE)
cleaned = process_data.clean_data(raw)
metrics = process_data.calculate_metrics(cleaned)

# Student-level rows only (drop the OVERALL summary row).
df = metrics[metrics["student_id"] != "OVERALL"].copy()
df["internship"] = df["internship"].astype(str).str.strip().str.lower()

print(f"Input records:    {len(raw)}")
print(f"Valid records:    {len(cleaned)}")
print(f"Excluded records: {len(raw) - len(cleaned)}")
display(df.head(8))

## 2. Data quality (guardrail metrics)

Errors and inconsistencies are tracked as guardrail metrics. A non-zero count would mean the data cannot be trusted for the statistics below.

In [ ]:
quality = statistical_analysis.summarize_data_quality(df)
display(quality.T.rename(columns={0: "value"}))

## 3. Statistical aggregation

For both outcome rates the notebook reports the mean, median, 95th and 99th percentiles, sample standard deviation, and a **bootstrap 95% confidence interval for the mean** (5000 resamples, seed 42). With n = 8, p95 and p99 are interpolated between very few points and are shown for completeness only.

In [ ]:
stats_df = statistical_analysis.summarize_metrics(df)
display(stats_df)

### Baseline vs. internship group

The exploratory comparison splits students by internship experience (baseline = no internship). Rates are in percent per student. This comparison is descriptive and does not establish causation.

In [ ]:
rows = []
for metric, label in [
    ("interview_invitation_rate", "Interview invitation rate"),
    ("job_offer_rate_per_application", "Job offer rate per application"),
]:
    for key, group_name in [("no", "No internship (baseline)"), ("yes", "Internship")]:
        values = df.loc[df["internship"] == key, metric].to_numpy(dtype=float) * 100
        rows.append({
            "indicator": label,
            "group": group_name,
            "n": len(values),
            "mean_%": values.mean(),
            "sd_%": values.std(ddof=1),
            "median_%": np.median(values),
            "p25_%": np.percentile(values, 25),
            "p75_%": np.percentile(values, 75),
        })

group_table = pd.DataFrame(rows)
display(group_table)

## 4. Visualization and export

The figures are produced by `src/visualize.py` and exported to `reports/figures/` as vector **PDF** and **PNG at 300 DPI**. They use a colorblind-safe palette; groups differ by marker, hatch and line style as well as colour. Axes are labelled with units (%), and uncertainty is shown as bootstrap 95% confidence intervals.

In [ ]:
visualize.plot_internship_comparison(df)
display(Image(filename=str(visualize.FIGURES_DIR / "fig1_internship_comparison.png"), width=600))

In [ ]:
visualize.plot_rate_distributions(df)
display(Image(filename=str(visualize.FIGURES_DIR / "fig2_rate_distributions.png"), width=650))

## 5. Save results

The summaries are written with the same functions and column layout as the `src/statistical_analysis.py` script, so running either one produces identical files.

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

stats_df.to_csv(statistical_analysis.SUMMARY_FILE, index=False)
quality.to_csv(statistical_analysis.QUALITY_FILE, index=False)

for path in [statistical_analysis.SUMMARY_FILE, statistical_analysis.QUALITY_FILE]:
    print("Saved:", path.relative_to(PROJECT_ROOT))
for name in ["fig1_internship_comparison", "fig2_rate_distributions"]:
    for ext in ("png", "pdf"):
        path = visualize.FIGURES_DIR / f"{name}.{ext}"
        print("Exists:", path.relative_to(PROJECT_ROOT), path.exists())

## 6. Interpretation and limitations

- The pipeline runs end to end: validation, descriptive statistics, bootstrap confidence intervals, guardrail checks and figure export.
- Students with internship experience show higher rates in this sample, but with four students per group the intervals are wide and the comparison is exploratory only.
- The sample is tiny (n = 17), so **no conclusion about H1, H2 or H3 can be drawn**; see `src/inferential_analysis.py` for the tests. Testing them requires real, appropriately collected student data and a suitable statistical analysis.